In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt


# Añade la carpeta raíz del proyecto al PATH
ROOT_DIR = Path.cwd().parent
#print(f"ROOT_DIR: {ROOT_DIR}")
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

# Import absoluto directo
from src.rl_project.envs.milan_taxy import MilanTaxiEnv
from src.rl_project.models.mdp import MDP, build_model
from src.rl_project.utils.utils import seed_everything, animate_frames, plot_gridworld_values
from src.rl_project.policies import RandomPolicy, rollout, ValueIterationPolicy

seed_everything(42)

env = MilanTaxiEnv()
mdp = MDP()



In [ ]:
from src.rl_project.utils.utils import seed_everything, animate_frames
from src.rl_project.policies import RandomPolicy, rollout

seed_everything(42)

env    = MilanTaxiEnv(6,6, max_steps=1000, p=0.8)
policy = RandomPolicy(actions_cardinality=env.action_space.n)

frames, _, _, _ = rollout(env, policy)

animate_frames(frames)

In [ ]:
nA = mdp.num_actions
nS = mdp.num_states
shape = (6,6)

In [ ]:

# P = mdp.P
# R = mdp.R

# assert np.allclose(P.sum(axis=2), 1.0), "each P[s, a, :] must be a probability distribution"
# print("P:", P.shape, " R:", R.shape, " -> all rows sum to 1")

In [ ]:
import numpy as np
env.num_states
s = np.ravel_multi_index((0, 0, 2, 0), (6, 6, 5, 4))
for a, name in enumerate(["SOUTH", "NORTH", "EAST", "WEST", "PICKUP", "DROPOFF"]):
    print(f"Action {a} {name} -> {mdp.P[s][a]}")

In [ ]:
import numpy as np
P, R = build_model(mdp)

assert np.allclose(P.sum(axis=2), 1.0), "each P[s, a, :] must be a probability distribution"
print("P:", P.shape, " R:", R.shape, " -> all rows sum to 1")

In [ ]:
def policy_matrices(P, R, pi):
    """Average the dynamics over the policy: returns (P_pi, R_pi)."""
    # Your code goes here: -------------------------------------
    P_pi = np.einsum('sa,sat->st', pi, P)
    R_pi = np.einsum('sa,sa->s', pi, R)
    return P_pi, R_pi
    # ----------------------------------------------------------


def solve_policy_direct(P, R, pi, gamma):
    """Exact V^pi by solving (I - gamma P^pi) V = R^pi."""
    P_pi, R_pi = policy_matrices(P, R, pi)
    # Your code goes here: -------------------------------------
    I = np.eye(P_pi.shape[0])
    V_pi = np.linalg.solve(I - gamma * P_pi, R_pi)
    return V_pi
    # ----------------------------------------------------------

GAMMA = 0.99
pi_random = np.ones((nS + 1, nA)) / nA          # uniform random policy

START = env.START
print(START)
# START = np.ravel_multi_index((0, 0, 2, 0), (6, 6, 5, 4))
V_random = solve_policy_direct(P, R, pi_random, GAMMA)
print(f"V^random(start) = {V_random[START]:.2f}")

In [ ]:
def iterative_policy_evaluation(P, R, pi, gamma, tol=1e-10, max_iter=100_000):
    P_pi, R_pi = policy_matrices(P, R, pi)
    V = np.zeros(P.shape[0])
    deltas = []

    for k in range(max_iter):
        # Your code goes here: -------------------------------------
        V_new = R_pi + gamma * P_pi @ V
        delta = np.max(np.abs(V_new - V))
        deltas.append(delta)
        V = V_new
        if delta < tol:
            break
        # ----------------------------------------------------------

    return V, k + 1, deltas


V_iter, n_sweeps, deltas = iterative_policy_evaluation(P, R, pi_random, GAMMA)

err = np.max(np.abs(V_iter - V_random))
print(f"{n_sweeps} sweeps  |  max |V_iterative - V_exact| = {err:.2e}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(deltas, label=r"$\|V_{k+1} - V_k\|_\infty$")
ax.semilogy(deltas[0] * GAMMA ** np.arange(len(deltas)), "--",
            label=rf"$\gamma^k$ rate ($\gamma = {GAMMA}$)")
ax.set_xlabel("sweep $k$")
ax.set_ylabel("change (log scale)")
ax.set_title("Iterative policy evaluation converges geometrically")
ax.legend()
ax.grid(alpha=.3)
plt.show()

In [ ]:
def q_from_v(P, R, V, gamma):
    """One-step lookahead: Q[s, a] from V."""
    # Your code goes here: -------------------------------------
    Q = R + gamma * P @ V
    return Q
    # ----------------------------------------------------------


def greedy_policy(Q):
    """Deterministic greedy policy, as a one-hot matrix."""
    # Your code goes here: -------------------------------------
    pi = np.zeros_like(Q)
    best_actions = np.argmax(Q, axis=1)
    pi[np.arange(Q.shape[0]), best_actions] = 1.0
    return pi
    # ----------------------------------------------------------


def policy_iteration(P, R, gamma, max_iter=1_000):
    pi = np.ones(R.shape) / R.shape[1]      # start from the random policy

    for k in range(max_iter):
        # Your code goes here: -------------------------------------
        V = iterative_policy_evaluation(P, R, pi, gamma)[0]
        pi = greedy_policy(q_from_v(P, R, V, gamma))   
        # ----------------------------------------------------------

    return V, pi, k + 1


V_star_pi, pi_star, n_iters = policy_iteration(P, R, GAMMA)
print(f"converged in {n_iters} iterations  |  V*(start) = {V_star_pi[START]:.2f}")

In [ ]:
# plot_gridworld_values(
#     V_star_pi[:nS].reshape(shape),
#     title="Optimal value function $V^*$ (policy iteration)",
#     cbar_label="$V^*(s)$",
# )

In [ ]:
def value_iteration(P, R, gamma, tol=1e-10, max_iter=100_000):
    V = np.zeros(R.shape[0])
    deltas = []

    for k in range(max_iter):
        # Your code goes here: -------------------------------------
        V_new = np.max(q_from_v(P, R, V, gamma), axis=1)
        delta = np.max(np.abs(V_new - V))
        deltas.append(delta)
        V = V_new
        if delta < tol:
            break
        # ----------------------------------------------------------

    pi = greedy_policy(q_from_v(P, R, V, gamma))
    return V, pi, k + 1, deltas


V_star_vi, pi_vi, n_sweeps_vi, deltas_vi = value_iteration(P, R, GAMMA)

print(f"value iteration:  {n_sweeps_vi} sweeps")
print(f"policy iteration: {n_iters} iterations")
print(f"max |V_VI - V_PI|   = {np.max(np.abs(V_star_vi - V_star_pi)):.2e}")
print(f"same greedy policy: {np.array_equal(pi_vi.argmax(1), pi_star.argmax(1))}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

for g in [0.5, 0.9, 0.99, 0.999]:
    _, _, _, d = value_iteration(P, R, g, tol=1e-12)
    ax.semilogy(d, label=rf"$\gamma = {g}$")

ax.set_xlabel("sweep $k$")
ax.set_ylabel(r"$\|V_{k+1} - V_k\|_\infty$")
ax.set_title("Convergence of value iteration vs. discount factor")
ax.legend()
ax.grid(alpha=.3)
plt.show()

In [ ]:
seed_everything(42)

policy = ValueIterationPolicy(policy=pi_vi, env=env)

frames, _, _, _ = rollout(env, policy)

animate_frames(frames)

In [ ]:
# import sys
# import timeit

# list = [1, 2, 3, 4, 5]
# tuple = (1, 2, 3, 4, 5)

# print(sys.getsizeof(list))  # Tamaño en bytes de la lista
# print(sys.getsizeof(tuple))  # Tamaño en bytes de la tupla

# print("List:  ",timeit.timeit(lambda: list[2], number=1000000))  # Tiempo para crear una list
# print("Tuple: ",timeit.timeit(lambda: tuple[2], number=1000000))  # Tiempo para crear una tupla
